In [2]:
from pathlib import Path
import sys

print("Python:", sys.executable)

Python: /home/bharadwaj/Workspace/india-legal-ai/backend/.venv/bin/python


In [3]:
from pathlib import Path

BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data"

print("Base directory:", BASE_DIR)
print("Data directory:", DATA_DIR)
print("Data exists:", DATA_DIR.exists())

Base directory: /home/bharadwaj/Workspace/india-legal-ai/backend/app
Data directory: /home/bharadwaj/Workspace/india-legal-ai/backend/app/data
Data exists: True


In [3]:
categories = [
    path.name
    for path in DATA_DIR.iterdir()
    if path.is_dir()
]

print("Legal categories:")
for category in categories:
    print(" -", category)

Legal categories:
 - technology
 - criminal
 - constitution_of_india
 - civil


In [4]:
pdf_files = list(DATA_DIR.rglob("*.pdf"))

print(f"Total PDF files: {len(pdf_files)}")

for pdf in pdf_files[:20]:
    print(pdf)

Total PDF files: 10
/home/bharadwaj/Workspace/india-legal-ai/backend/app/data/technology/information_technology_act.pdf
/home/bharadwaj/Workspace/india-legal-ai/backend/app/data/criminal/bns_2023.pdf
/home/bharadwaj/Workspace/india-legal-ai/backend/app/data/criminal/crpc_1973.pdf
/home/bharadwaj/Workspace/india-legal-ai/backend/app/data/criminal/evidence_act_1872.pdf
/home/bharadwaj/Workspace/india-legal-ai/backend/app/data/criminal/bsa_2023.pdf
/home/bharadwaj/Workspace/india-legal-ai/backend/app/data/criminal/bnss_2023.pdf
/home/bharadwaj/Workspace/india-legal-ai/backend/app/data/criminal/ipc_1860.pdf
/home/bharadwaj/Workspace/india-legal-ai/backend/app/data/constitution_of_india/constitution_of_india.pdf
/home/bharadwaj/Workspace/india-legal-ai/backend/app/data/civil/transfer_of_property_act.pdf
/home/bharadwaj/Workspace/india-legal-ai/backend/app/data/civil/contract_act.pdf


In [6]:
import pymupdf

print("PyMuPDF:", pymupdf.__version__)

PyMuPDF: 1.28.2


In [8]:
pdf_path = pdf_files[0]

print("Selected PDF:")
print(pdf_path)

Selected PDF:
/home/bharadwaj/Workspace/india-legal-ai/backend/app/data/technology/information_technology_act.pdf


In [9]:
doc = pymupdf.open(pdf_path)

print("File:", pdf_path.name)
print("Number of pages:", len(doc))

File: information_technology_act.pdf
Number of pages: 36


In [11]:
page = doc[0]

text = page.get_text()

print(text[:5000])

1 
 
THE INFORMATION TECHNOLOGY ACT, 2000 
––––––––– 
ARRANGEMENT OF SECTIONS 
––––––––– 
CHAPTER I 
PRELIMINARY 
SECTIONS 
1. Short title, extent, commencement and application. 
2. Definitions. 
CHAPTER II 
DIGITAL SIGNATURE AND ELECTRONIC SIGNATURE 
3. Authentication of electronic records. 
3A. Electronic signature. 
CHAPTER III 
ELECTRONIC GOVERNANCE 
4. Legal recognition of electronic records. 
5. Legal recognition of electronic signatures. 
6. Use of electronic records and electronic signatures in Government and its agencies. 
6A. Delivery of services by service provider. 
7. Retention of electronic records. 
7A. Audit of documents, etc., maintained in electronic form. 
8. Publication of rule, regulation, etc., in Electronic Gazette. 
9. Sections 6, 7 and 8 not to confer right to insist document should be accepted in electronic form. 
10. Power to make rules by Central Government in respect of electronic signature. 
10A. Validity of contracts formed through electronic means. 
CHAP

In [12]:
def extract_pdf_pages(pdf_path):
    doc = fitz.open(pdf_path)

    pages = []

    for page_number, page in enumerate(doc, start=1):
        text = page.get_text()

        pages.append({
            "page_number": page_number,
            "text": text
        })

    doc.close()

    return pages

In [13]:
pages = extract_pdf_pages(pdf_path)

print("Pages extracted:", len(pages))

Pages extracted: 36


In [14]:
for page in pages[:3]:
    print("=" * 80)
    print("PAGE:", page["page_number"])
    print(page["text"][:2000])

PAGE: 1
1 
 
THE INFORMATION TECHNOLOGY ACT, 2000 
––––––––– 
ARRANGEMENT OF SECTIONS 
––––––––– 
CHAPTER I 
PRELIMINARY 
SECTIONS 
1. Short title, extent, commencement and application. 
2. Definitions. 
CHAPTER II 
DIGITAL SIGNATURE AND ELECTRONIC SIGNATURE 
3. Authentication of electronic records. 
3A. Electronic signature. 
CHAPTER III 
ELECTRONIC GOVERNANCE 
4. Legal recognition of electronic records. 
5. Legal recognition of electronic signatures. 
6. Use of electronic records and electronic signatures in Government and its agencies. 
6A. Delivery of services by service provider. 
7. Retention of electronic records. 
7A. Audit of documents, etc., maintained in electronic form. 
8. Publication of rule, regulation, etc., in Electronic Gazette. 
9. Sections 6, 7 and 8 not to confer right to insist document should be accepted in electronic form. 
10. Power to make rules by Central Government in respect of electronic signature. 
10A. Validity of contracts formed through electronic mean

In [15]:
total_characters = sum(
    len(page["text"])
    for page in pages
)

print("Total characters:", total_characters)

Total characters: 127829


In [16]:
empty_pages = [
    page["page_number"]
    for page in pages
    if not page["text"].strip()
]

print("Empty pages:", len(empty_pages))
print("Empty page numbers:", empty_pages[:20])

Empty pages: 0
Empty page numbers: []


In [17]:
import re


def parse_legal_document(pages, document, category):
    """
    Convert extracted PDF pages into structured legal chunks.

    Parameters
    ----------
    pages : list[dict]
        Output from extract_pdf_pages().
    document : str
        Document / Act name.
    category : str
        Legal category, e.g. criminal, civil.

    Returns
    -------
    list[dict]
        Structured legal chunks.
    """

    chunks = []

    current_chapter = None
    current_section = None
    current_subsection = None
    current_text = []
    current_page = None

    def save_chunk():
        if not current_text:
            return

        text = "\n".join(current_text).strip()

        if not text:
            return

        chunks.append({
            "text": text,
            "page_number": current_page,
            "section": current_section,
            "subsection": current_subsection,
            "chapter": current_chapter,
            "document": document,
            "category": category
        })

    for page in pages:
        page_number = page["page_number"]
        text = page["text"]

        lines = text.splitlines()

        for line in lines:
            line = line.strip()

            if not line:
                continue

            # Chapter
            chapter_match = re.match(
                r"^(CHAPTER\s+[IVXLCDM0-9]+.*)$",
                line,
                re.IGNORECASE
            )

            if chapter_match:
                current_chapter = chapter_match.group(1).strip()
                continue

            # Section
            section_match = re.match(
                r"^(SECTION\s+\d+[A-Z]?)\b(.*)$",
                line,
                re.IGNORECASE
            )

            if section_match:
                save_chunk()

                current_section = section_match.group(1).strip()
                current_subsection = None
                current_text = [line]
                current_page = page_number

                continue

            # Subsection: (1), (2), (a), (b), etc.
            subsection_match = re.match(
                r"^(\([0-9A-Za-z]+\))\s*(.*)$",
                line
            )

            if subsection_match and current_section:
                save_chunk()

                current_subsection = subsection_match.group(1)

                content = subsection_match.group(2).strip()

                current_text = [
                    f"{current_subsection} {content}"
                ]

                current_page = page_number

                continue

            # Normal legal text
            if current_text:
                current_text.append(line)
            else:
                current_text = [line]
                current_page = page_number

    # Save final chunk
    save_chunk()

    return chunks

In [18]:
pages = extract_pdf_pages(pdf_path)

In [19]:
chunks = parse_legal_document(
    pages=pages,
    document=pdf_path.stem,
    category=pdf_path.parent.name
)

In [21]:
print("Chunks:", len(chunks))

Chunks: 379


In [22]:
chunks[0]

{'text': '1\nTHE INFORMATION TECHNOLOGY ACT, 2000\n–––––––––\nARRANGEMENT OF SECTIONS\n–––––––––\nPRELIMINARY\nSECTIONS\n1. Short title, extent, commencement and application.\n2. Definitions.\nDIGITAL SIGNATURE AND ELECTRONIC SIGNATURE\n3. Authentication of electronic records.\n3A. Electronic signature.\nELECTRONIC GOVERNANCE\n4. Legal recognition of electronic records.\n5. Legal recognition of electronic signatures.\n6. Use of electronic records and electronic signatures in Government and its agencies.\n6A. Delivery of services by service provider.\n7. Retention of electronic records.\n7A. Audit of documents, etc., maintained in electronic form.\n8. Publication of rule, regulation, etc., in Electronic Gazette.\n9. Sections 6, 7 and 8 not to confer right to insist document should be accepted in electronic form.\n10. Power to make rules by Central Government in respect of electronic signature.\n10A. Validity of contracts formed through electronic means.\nATTRIBUTION, ACKNOWLEDGEMENT AND

In [24]:
all_chunks = []

for pdf_path in pdf_files:
    try:
        pages = extract_pdf_pages(pdf_path)

        chunks = parse_legal_document(
            pages=pages,
            document=pdf_path.stem,
            category=pdf_path.parent.name
        )

        all_chunks.extend(chunks)

        print(
            f"{pdf_path.name}: "
            f"{len(pages)} pages → {len(chunks)} chunks"
        )

    except Exception as e:
        print(f"ERROR: {pdf_path}")
        print(e)

print("\nTotal chunks:", len(all_chunks))

information_technology_act.pdf: 36 pages → 379 chunks
bns_2023.pdf: 102 pages → 610 chunks
crpc_1973.pdf: 368 pages → 1505 chunks
evidence_act_1872.pdf: 60 pages → 270 chunks
bsa_2023.pdf: 47 pages → 230 chunks
bnss_2023.pdf: 249 pages → 1525 chunks
ipc_1860.pdf: 258 pages → 335 chunks
constitution_of_india.pdf: 268 pages → 768 chunks
transfer_of_property_act.pdf: 46 pages → 174 chunks
contract_act.pdf: 53 pages → 266 chunks

Total chunks: 6062


In [27]:
print("Total chunks:", len(all_chunks))
print("Chunks with section:", sum(1 for c in all_chunks if c["section"]))
print("Chunks with chapter:", sum(1 for c in all_chunks if c["chapter"]))
print("Chunks with subsection:", sum(1 for c in all_chunks if c["subsection"]))

Total chunks: 6062
Chunks with section: 6052
Chunks with chapter: 5805
Chunks with subsection: 5818


In [30]:
import pandas as pd

chunk_df = pd.DataFrame(all_chunks)

chunk_df["text_length"] = chunk_df["text"].str.len()

print("Total chunks:", len(chunk_df))
print()
print(chunk_df["text_length"].describe())

Total chunks: 6062

count      6062.000000
mean        638.625371
std        3637.223987
min           3.000000
25%         125.000000
50%         246.000000
75%         554.000000
max      190632.000000
Name: text_length, dtype: float64


In [31]:
large_chunks = chunk_df[
    chunk_df["text_length"] > 3000
].sort_values(
    "text_length",
    ascending=False
)

print("Chunks > 3000 chars:", len(large_chunks))

large_chunks[
    [
        "document",
        "category",
        "chapter",
        "section",
        "subsection",
        "page_number",
        "text_length"
    ]
].head(20)

Chunks > 3000 chars: 116


,document,category,chapter,section,subsection,page_number,text_length
4854,constitution_of_india,constitution_of_india,CHAPTER III.—THE STATE LEGISLATURE,NaN,NaN,1,190632
4519,ipc_1860,criminal,CHAPTER VIII - OF OFFENCES AGAINST THE PUBLIC ...,NaN,NaN,1,152168
2288,crpc_1973,criminal,CHAPTER XVIII-OFFENCES RELATING TO DOCUMENTS A...,section 161,(b),276,56817
2764,bsa_2023,criminal,CHAPTER II,NaN,NaN,1,55339
379,bns_2023,criminal,NaN,NaN,NaN,1,52153
4422,bnss_2023,criminal,CHAPTER XXXIX,section 210,(2),158,50068
2994,bnss_2023,criminal,CHAPTER III,NaN,NaN,1,32299
4555,ipc_1860,criminal,CHAPTER XI - OF FALSE EVIDENCE AND OFFENCES AG...,section 141,(SC),105,29579
4426,bnss_2023,criminal,CHAPTER XXXIX,section 84,(b),179,26413
2286,crpc_1973,criminal,CHAPTER XI-FALSE EVIDENCE AND OFFENCES AGAINST...,section 161,NaN,262,24876


In [34]:
largest = large_chunks.iloc[0]

print("Document:", largest["document"])
print("Section:", largest["section"])
print("Subsection:", largest["subsection"])
print("Page:", largest["page_number"])
print("Length:", largest["text_length"])

Document: constitution_of_india
Section: nan
Subsection: nan
Page: 1
Length: 190632
